In [20]:
import pandas as pd
import numpy as np
import duckdb as db
import requests
import io
import zipfile


# Import liquor sales data (2025 + 2026) + convert to parquet

In [21]:
con = db.connect()

con.execute("""
    COPY (
        SELECT *
        FROM read_json_auto(
            'iowa_liquor_sales_2025_1262_rows/**/*.json'
        )
    )
    TO 'liquor_2025.parquet' (FORMAT PARQUET)
""")

con.execute("""
    COPY (
        SELECT *
        FROM read_json_auto(
            'iowa_liquor_sales_2026_1263_rows/**/*.json'
        )
    )
    TO 'liquor_2026.parquet' (FORMAT PARQUET)
""")

df_2025 = pd.read_parquet("liquor_2025.parquet")
df_2026 = pd.read_parquet("liquor_2026.parquet")

df_2025["ordered_on"] = pd.to_datetime(df_2025["ordered_on"])
df_2026["ordered_on"] = pd.to_datetime(df_2026["ordered_on"])

print(df_2025.shape, df_2026.shape)
df_2025.head()


(2907817, 23) (1968354, 23)


,invoice_id,ordered_on,store_no,store_name,store_address,store_city,store_zip_code,county_fips_code,county_name,category_code,...,item_no,im_desc,pack,bottle_volume_ml,sales_bottles,sales_dollars,sales_liters,sales_gallons,state_bottle_cost,state_bottle_retail
0,INV-78192100069,2025-01-01,2505,HY-VEE WINE AND SPIRITS (1038) / BOONE,1111 8TH ST,BOONE,50036,19015,BOONE,1022200,...,87510,1800 SILVER,12,750,3,74.25,2.25,0.59,NaN,NaN
1,INV-78192100072,2025-01-01,2505,HY-VEE WINE AND SPIRITS (1038) / BOONE,1111 8TH ST,BOONE,50036,19015,BOONE,1022200,...,88540,HORNITOS LIME SHOT,12,750,12,270.00,9.00,2.37,NaN,NaN
2,INV-78192100070,2025-01-01,2505,HY-VEE WINE AND SPIRITS (1038) / BOONE,1111 8TH ST,BOONE,50036,19015,BOONE,1022200,...,88036,MARGARITAVILLE SILVER TEQUILA,12,750,3,41.85,2.25,0.59,NaN,NaN
3,INV-78192100071,2025-01-01,2505,HY-VEE WINE AND SPIRITS (1038) / BOONE,1111 8TH ST,BOONE,50036,19015,BOONE,1022200,...,88172,SANTO TEQUILA REPOSADO,6,750,2,82.50,1.50,0.39,NaN,NaN
4,INV-78192100061,2025-01-01,2505,HY-VEE WINE AND SPIRITS (1038) / BOONE,1111 8TH ST,BOONE,50036,19015,BOONE,1081300,...,85526,DEKUYPER BLUE CURACAO,12,750,12,94.56,9.00,2.37,NaN,NaN


In [22]:
df_2026.head()

,invoice_id,ordered_on,store_no,store_name,store_address,store_city,store_zip_code,county_fips_code,county_name,category_code,...,item_no,im_desc,pack,bottle_volume_ml,state_bottle_cost,state_bottle_retail,sales_bottles,sales_dollars,sales_liters,sales_gallons
0,INV-903316,2026-01-01,5444,MARSHALL BEER WINE SPIRITS,11 N 3RD AVE,MARSHALLTOWN,50158,19127,MARSHALL,1022200,...,88296,PATRON SILVER,12,750,24.99,37.49,60,2249.40,45.0,11.887740
1,INV-903316,2026-01-01,5444,MARSHALL BEER WINE SPIRITS,11 N 3RD AVE,MARSHALLTOWN,50158,19127,MARSHALL,1082200,...,69947,RUMPLE MINZE PEPPERMINT SCHNAPPS LIQUEUR,12,1000,16.99,25.49,36,917.64,36.0,9.510192
2,INV-903479,2026-01-02,4417,MAVERIK #5066 / ADAIR,109 S 5TH ST,ADAIR,50002,19001,ADAIR,1022200,...,87484,DON JULIO BLANCO,12,375,14.00,22.00,3,67.00,1.0,0.000000
3,INV-903479,2026-01-02,4417,MAVERIK #5066 / ADAIR,109 S 5TH ST,ADAIR,50002,19001,ADAIR,1022200,...,88296,PATRON SILVER,12,750,24.00,37.00,12,449.00,9.0,2.000000
4,INV-903479,2026-01-02,4417,MAVERIK #5066 / ADAIR,109 S 5TH ST,ADAIR,50002,19001,ADAIR,1081300,...,84393,99 WATERMELON PET MINI,1,50,54.00,81.00,1,81.00,0.0,0.000000


# Pull accompanying Iowa weather data (per zip code)

Same nearest-station idea as before, but keyed on `store_zip_code` instead
of county so it lines up with a `merge()` directly against `df_2025`/`df_2026`:

1. Pull the list of Iowa ASOS stations (`IA_ASOS` network, 62 of them) from IEM.
2. Pull ZCTA (zip code) centroid coordinates from the Census Bureau gazetteer, filtered to just the zip codes that actually show up in the liquor data.
3. Assign each zip code its nearest ASOS station by straight-line distance.
4. Fetch daily weather for every station used, and build `weather_df_2025` / `weather_df_2026` as one row per zip code per day — ready to `merge()` onto `df_2025`/`df_2026` on `["store_zip_code", "ordered_on"]`.
5. Compute a statewide daily average across all stations as a fallback, and use it for any liquor sales row whose zip code is missing, blank, or doesn't resolve to an Iowa zip (this is exactly the same set of rows that have an unknown/blank `county_name` — the two are missing together in this data).

In [23]:
def get_iowa_asos_stations():
    """Iowa ASOS station metadata (id, station's home county, lat/lon)."""
    url = "https://mesonet.agron.iastate.edu/geojson/network/IA_ASOS.geojson"
    data = requests.get(url, timeout=30).json()

    rows = []
    for feat in data["features"]:
        props = feat["properties"]
        lon, lat = feat["geometry"]["coordinates"]
        rows.append({
            "station": props["sid"],
            "station_name": props["sname"],
            "county": props["county"].upper(),
            "lat": lat,
            "lon": lon,
            "online": props["online"],
        })

    stations_df = pd.DataFrame(rows)
    return stations_df[stations_df["online"]].reset_index(drop=True)


def get_zip_centroids(zips_needed):
    """Centroid lat/lon for a set of 5-digit zip codes (Census ZCTA gazetteer).

    The gazetteer is only published as a .zip on census.gov -- there is no
    plain-text URL for it (a bare .txt request 404s/520s), so download the
    archive and read the .txt out of it in memory.
    """
    url = "https://www2.census.gov/geo/docs/maps-data/data/gazetteer/2024_Gazetteer/2024_Gaz_zcta_national.zip"
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        name = next(n for n in zf.namelist() if n.endswith(".txt"))
        with zf.open(name) as fh:
            zips_df = pd.read_csv(fh, sep="\t", dtype={"GEOID": str}, encoding="latin-1")

    zips_df.columns = [c.strip() for c in zips_df.columns]
    zips_df["zip"] = zips_df["GEOID"].str.zfill(5)
    zips_df = zips_df.rename(columns={"INTPTLAT": "lat", "INTPTLONG": "lon"})

    return zips_df.loc[zips_df["zip"].isin(zips_needed), ["zip", "lat", "lon"]].reset_index(drop=True)


def haversine_miles(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 3958.8 * 2 * np.arcsin(np.sqrt(a))


def clean_zip(series):
    z = series.astype(str).str.strip().str.extract(r"(\d{5})")[0]
    return z


stations_df = get_iowa_asos_stations()

# every zip code that actually shows up in the liquor data (blanks/NaNs dropped here on purpose --
# those rows get the statewide average fallback later)
zips_needed = sorted(
    set(clean_zip(df_2025["store_zip_code"]).dropna())
    | set(clean_zip(df_2026["store_zip_code"]).dropna())
)

zip_centroids_df = get_zip_centroids(zips_needed)
unmatched_zips = sorted(set(zips_needed) - set(zip_centroids_df["zip"]))

zip_to_station = {}
for _, row in zip_centroids_df.iterrows():
    dists = haversine_miles(row["lat"], row["lon"], stations_df["lat"], stations_df["lon"])
    zip_to_station[row["zip"]] = stations_df.loc[dists.idxmin(), "station"]

zip_station_df = pd.DataFrame(zip_to_station.items(), columns=["store_zip_code", "station"])

print(f"{len(zip_to_station)} / {len(zips_needed)} zip codes matched to a station")
print(f"{len(unmatched_zips)} zip codes had no centroid on file (will fall back to statewide avg): {unmatched_zips[:10]}")
print(f"{zip_station_df['station'].nunique()} unique stations used")
zip_station_df.head()

507 / 512 zip codes matched to a station
5 zip codes had no centroid on file (will fall back to statewide avg): ['52087', '52303', '52671', '52733', '57222']
61 unique stations used


,store_zip_code,station
0,50002,ADU
1,50003,PRO
2,50005,MIW
3,50006,IFA
4,50008,CNC


In [24]:
def fetch_iem_daily_weather(stations, start_date, end_date, network="IA_ASOS"):
    """Fetch daily weather summaries from the Iowa Environmental Mesonet."""
    start_date = pd.Timestamp(start_date)
    end_date = pd.Timestamp(end_date)

    url = "https://mesonet.agron.iastate.edu/cgi-bin/request/daily.py"
    params = {
        "network": network,
        "stations": ",".join(sorted(set(stations))),
        "year1": start_date.year,
        "month1": start_date.month,
        "day1": start_date.day,
        "year2": end_date.year,
        "month2": end_date.month,
        "day2": end_date.day,
        "format": "csv",
    }

    resp = requests.get(url, params=params, timeout=120)
    resp.raise_for_status()

    weather_df = pd.read_csv(io.StringIO(resp.text))
    weather_df["day"] = pd.to_datetime(weather_df["day"])
    return weather_df


all_stations = zip_station_df["station"].unique()

station_weather_2025 = fetch_iem_daily_weather(
    all_stations, df_2025["ordered_on"].min(), df_2025["ordered_on"].max()
)
station_weather_2026 = fetch_iem_daily_weather(
    all_stations, df_2026["ordered_on"].min(), df_2026["ordered_on"].max()
)

weather_cols = [c for c in station_weather_2025.columns if c not in ("station", "day")]

# one row per zip code per day
weather_df_2025 = zip_station_df.merge(station_weather_2025, on="station", how="left")
weather_df_2026 = zip_station_df.merge(station_weather_2026, on="station", how="left")

# statewide daily average across every station in use -- the imputed fallback
statewide_avg_2025 = station_weather_2025.groupby("day")[weather_cols].mean().reset_index()
statewide_avg_2026 = station_weather_2026.groupby("day")[weather_cols].mean().reset_index()

print(weather_df_2025.shape, weather_df_2026.shape)
weather_df_2025.head()

(185055, 24) (123201, 24)


,store_zip_code,station,day,max_temp_f,min_temp_f,max_dewpoint_f,min_dewpoint_f,precip_in,avg_wind_speed_kts,avg_wind_drct,...,snowd_in,min_feel,avg_feel,max_feel,max_wind_speed_kts,max_wind_gust_kts,srad_mj,climo_high_f,climo_low_f,climo_precip_in
0,50002,ADU,2025-01-01,33.8,19.4,26.6,17.6,0.00,4.006969,288.939850,...,NaN,16.799290,22.657082,33.800000,10.0,16.0,NaN,28.7,10.5,0.03
1,50002,ADU,2025-01-02,26.6,10.4,24.8,8.6,0.00,3.832753,28.087036,...,NaN,7.156008,17.615230,26.600000,9.0,NaN,NaN,28.5,10.3,0.03
2,50002,ADU,2025-01-03,21.2,8.6,19.4,6.8,0.06,4.017422,317.492580,...,NaN,-1.549657,8.698599,17.600000,9.0,NaN,NaN,28.4,10.1,0.03
3,50002,ADU,2025-01-04,15.8,8.6,8.6,-0.4,0.00,6.665505,34.344624,...,NaN,-5.489315,-0.105899,8.600000,12.0,16.0,NaN,28.3,9.9,0.03
4,50002,ADU,2025-01-05,15.8,10.4,6.8,-2.2,0.00,12.414634,2.522594,...,NaN,-7.735320,-3.445042,3.370198,18.0,23.0,NaN,28.1,9.8,0.03


## Join weather onto the liquor sales data

`attach_weather` merges on `store_zip_code` + date, then fills any row that
didn't get a match (missing/blank zip, unknown county, out-of-state or
untracked zip) with that day's statewide average instead of leaving it
blank.

In [25]:
def attach_weather(df, weather_df, statewide_avg, weather_cols):
    """Left-join weather onto liquor sales by (store_zip_code, ordered_on),
    then fill any unmatched row (missing/blank/out-of-state zip -- which in
    this data is exactly the unknown-county rows too) with that day's
    statewide average."""
    df = df.copy()
    df["store_zip_code"] = clean_zip(df["store_zip_code"])

    weather_keyed = weather_df.drop(columns="station").rename(columns={"day": "ordered_on"})
    merged = df.merge(weather_keyed, on=["store_zip_code", "ordered_on"], how="left")

    statewide_keyed = statewide_avg.rename(columns={"day": "ordered_on"})
    statewide_keyed = statewide_keyed.rename(columns={c: f"{c}_statewide" for c in weather_cols})
    merged = merged.merge(statewide_keyed, on="ordered_on", how="left")

    for col in weather_cols:
        merged[col] = merged[col].fillna(merged[f"{col}_statewide"])

    return merged.drop(columns=[f"{c}_statewide" for c in weather_cols])


df_2025_weather = attach_weather(df_2025, weather_df_2025, statewide_avg_2025, weather_cols)
df_2026_weather = attach_weather(df_2026, weather_df_2026, statewide_avg_2026, weather_cols)

print(df_2025_weather.shape, df_2026_weather.shape)
print("rows still missing weather after fallback:", df_2025_weather[weather_cols[0]].isna().sum())
df_2025_weather.head()

(2907817, 44) (1968354, 44)
rows still missing weather after fallback: 0


,invoice_id,ordered_on,store_no,store_name,store_address,store_city,store_zip_code,county_fips_code,county_name,category_code,...,snowd_in,min_feel,avg_feel,max_feel,max_wind_speed_kts,max_wind_gust_kts,srad_mj,climo_high_f,climo_low_f,climo_precip_in
0,INV-78192100069,2025-01-01,2505,HY-VEE WINE AND SPIRITS (1038) / BOONE,1111 8TH ST,BOONE,50036,19015,BOONE,1022200,...,0.00002,8.635962,17.926886,24.8,23.0,30.0,NaN,29.2,11.2,0.04
1,INV-78192100072,2025-01-01,2505,HY-VEE WINE AND SPIRITS (1038) / BOONE,1111 8TH ST,BOONE,50036,19015,BOONE,1022200,...,0.00002,8.635962,17.926886,24.8,23.0,30.0,NaN,29.2,11.2,0.04
2,INV-78192100070,2025-01-01,2505,HY-VEE WINE AND SPIRITS (1038) / BOONE,1111 8TH ST,BOONE,50036,19015,BOONE,1022200,...,0.00002,8.635962,17.926886,24.8,23.0,30.0,NaN,29.2,11.2,0.04
3,INV-78192100071,2025-01-01,2505,HY-VEE WINE AND SPIRITS (1038) / BOONE,1111 8TH ST,BOONE,50036,19015,BOONE,1022200,...,0.00002,8.635962,17.926886,24.8,23.0,30.0,NaN,29.2,11.2,0.04
4,INV-78192100061,2025-01-01,2505,HY-VEE WINE AND SPIRITS (1038) / BOONE,1111 8TH ST,BOONE,50036,19015,BOONE,1081300,...,0.00002,8.635962,17.926886,24.8,23.0,30.0,NaN,29.2,11.2,0.04
